# 🌲 Wildfire Smoke Early-Warning Detector: MobileNetV2 Transfer Learning
This Google Colab notebook trains an optimized **MobileNetV2** deep learning model for detecting smoke from forest surveillance cameras.

### Pipeline Steps:
1. **GPU Runtime Setup**
2. **Dataset Acquisition & Preprocessing** (Wildfire Smoke vs. Forest/Clouds)
3. **Data Augmentation** (handling lighting variations, smoke dispersion, weather)
4. **MobileNetV2 Transfer Learning Architecture**
5. **Training with Early Stopping & ReduceLROnPlateau**
6. **Evaluation**: Precision, Recall, F1-Score, Confusion Matrix
7. **Export**: Save `.keras` model for the Flask web application

In [ ]:
# Check GPU Availability
!nvidia-smi

In [ ]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

print(f"TensorFlow Version: {tf.__version__}")

### Step 2: Download or Mount Dataset
Organize your dataset into two classes: `smoke` and `no_smoke`.
You can upload a zip file or connect your Google Drive.

In [ ]:
# Example: Mount Google Drive or upload dataset.zip
# from google.colab import drive
# drive.mount('/content/drive')

# Or unzip uploaded dataset
!mkdir -p dataset/smoke dataset/no_smoke
print("Ensure images are placed in /content/dataset/smoke and /content/dataset/no_smoke")

### Step 3: Dataset Splitting & Preprocessing

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
DATA_DIR = '/content/dataset'

train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset='training',
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset='validation',
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

class_names = train_ds.class_names
print(f"Classes: {class_names}")

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)

### Step 4: Build MobileNetV2 Model with Augmentation

In [ ]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1)
], name="data_augmentation")

# MobileNetV2 Backbone
base_model = MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False

inputs = layers.Input(shape=(224, 224, 3))
x = data_augmentation(inputs)
x = layers.Rescaling(1./255)(x)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)
x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.35)(x)
outputs = layers.Dense(1, activation='sigmoid')(x)

model = models.Model(inputs=inputs, outputs=outputs, name="WildfireSmoke_MobileNetV2")
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.Precision(name='precision'), tf.keras.metrics.Recall(name='recall')]
)
model.summary()

### Step 5: Train Model with Early Stopping

In [ ]:
callbacks = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
    ModelCheckpoint('smoke_detector_mobilenetv2.keras', monitor='val_loss', save_best_only=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6, verbose=1)
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=callbacks
)

### Step 6: Plot Loss and Accuracy

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history['accuracy'], label='Train')
axes[0].plot(history.history['val_accuracy'], label='Val')
axes[0].set_title('Accuracy')
axes[0].legend()

axes[1].plot(history.history['loss'], label='Train')
axes[1].plot(history.history['val_loss'], label='Val')
axes[1].set_title('Loss')
axes[1].legend()
plt.show()

### Step 7: Download Trained Model for the Flask App
Download `smoke_detector_mobilenetv2.keras` and place it in your project's `models/` directory!

In [ ]:
from google.colab import files
files.download('smoke_detector_mobilenetv2.keras')